In [1]:
import prepocess, importlib
importlib.reload(prepocess)
from prepocess import preprocess
import pandas as pd


file_path = "syn_data/syn_net_p0.8_mu0.2_1.csv"
node2id, num_nodes = preprocess(file_path=file_path)

50 nodes in the link stream


/Users/acw721/Desktop/research/linkstream/prepocess.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  link_stream = pd.read_csv(


In [2]:
link_stream = pd.read_csv('preprocessed/syn_net_p0.8_mu0.2_1_pcs.csv')

In [3]:
from tgn.utils.my_data import get_data  
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

data = 'syn_net_p0.8_mu0.2_pcs'

node_feat, edge_feat, full_data = get_data(data, "preprocessed/syn_net_p0.8_mu0.2_1_pcs.csv", node_embedding_method="one-hot")

max_idx = max(full_data.unique_nodes)

cannot find (./data/syn_net_p0.8_mu0.2_pcs.npy), use zero-vector for edge feat (dim=16)...
Use one-hot init for node embedding. 
The dataset has 469 interactions, involving 50 different nodes


In [ ]:
from tgn.utils.utils import EarlyStopMonitor, get_neighbor_finder, MLP

ngh_finder = get_neighbor_finder(full_data, uniform=True, max_node_idx=max_idx)

In [5]:
from tgn.utils.my_data import get_data, compute_time_statistics, compute_time_statistics_undirected
import importlib
import sys
importlib.reload(sys.modules['tgn.utils.my_data'])

mean_time_shift, std_time_shift= compute_time_statistics_undirected(full_data.sources, 
                                full_data.destinations, 
                                full_data.timestamps)


In [33]:
importlib.reload(sys.modules['tgn.model.my_tgn'])
from tgn.model.my_tgn import TGN

from pathlib import Path


import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path
import torch


NUM_NEIGHBORS = 20
NUM_HEADS = 4
DROP_OUT = 0.1
NUM_LAYER = 2
LEARNING_RATE = 0.001
NODE_DIM = node_feat.shape[1]
TIME_DIM = 16
USE_MEMORY = True
MESSAGE_DIM = 100
MEMORY_DIM = NODE_DIM
num_communities = 5
device = 'cuda' if torch.cuda.is_available() else 'mps'
prefix = 'syn_net'
aggregator = 'mean'
memory_update_at_end = False
embedding_module = 'graph_attention' # graph_attention, graph_sum, time, identity
message_function = 'mlp'


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


tgn = TGN(
    neighbor_finder=ngh_finder,
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    n_layers=NUM_LAYER,
    n_heads=NUM_HEADS,
    dropout=DROP_OUT,
    use_memory=USE_MEMORY,
    message_dimension=MESSAGE_DIM,
    memory_dimension=MEMORY_DIM,
    n_neighbors=NUM_NEIGHBORS,
    mean_time_shift=mean_time_shift,
    std_time_shift=std_time_shift,
    use_destination_embedding_in_message=True,
    use_source_embedding_in_message=True,
    memory_update_at_start=not memory_update_at_end,
    embedding_module_type=embedding_module,
    message_function=message_function,
    aggregator_type=aggregator, 
    num_communities=num_communities
).to(device)

optimizer = torch.optim.Adam(tgn.parameters(), lr=LEARNING_RATE)

BATCH_SIZE = 500

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
Path("log/").mkdir(parents=True, exist_ok=True)
fh = logging.FileHandler('log/{}.log'.format(str(time.time())))
fh.setLevel(logging.DEBUG)
ch = logging.StreamHandler()
ch.setLevel(logging.WARN)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)


Path("./saved_models/").mkdir(parents=True, exist_ok=True)
Path("./saved_checkpoints/").mkdir(parents=True, exist_ok=True)
MODEL_SAVE_PATH = f'./saved_models/{prefix}-{data}.pth'
get_checkpoint_path = lambda \
    epoch: f'./saved_checkpoints/{prefix}-{data}-{epoch}.pth'


num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
logger.debug(f'num_batches: {num_batches}')

DEBUG:root:num_batches: 1


In [4]:

from pathlib import Path
import math
import logging
import time
import sys
import argparse
import numpy as np
import pickle
from pathlib import Path
import torch


BATCH_SIZE = 500

num_instance = len(full_data.sources)
num_batches = math.ceil(len(full_data.sources) / BATCH_SIZE)
print(f'num_batches: {num_batches}')

num_batches: 1


In [5]:
import torch

class ExpKernelActivity:
    """
    Maintain k_u(t) = sum_{events at times t_i < t} exp(-beta*(t - t_i))
    with lazy decay.

    - query(nodes, t): returns k(nodes, t) WITHOUT modifying the stored state
    - update_endpoints(src, dst, t): applies decay-to-t and then increments by 1 for endpoints
    """
    def __init__(self, num_nodes, beta, device, dtype=torch.float32, t0=0.0):
        self.beta = float(beta)
        self.device = device
        self.k = torch.zeros(num_nodes, device=device, dtype=dtype)
        self.last_t = torch.full((num_nodes,), float(t0), device=device, dtype=dtype)

    def _decay_to(self, nodes: torch.Tensor, t: torch.Tensor):
        """
        In-place: decay k[nodes] from last_t to time t (t can be scalar or [N]).
        """
        # t shape: scalar or [N]
        last = self.last_t[nodes]
        dt = torch.clamp(t.to(last.dtype) - last, min=0.0)
        if self.beta > 0:
            self.k[nodes] = self.k[nodes] * torch.exp(-self.beta * dt)
        # update last_t to current t
        self.last_t[nodes] = t.to(last.dtype)

    @torch.no_grad()
    def update_endpoints(self, src: torch.Tensor, dst: torch.Tensor, t: torch.Tensor, inc=1.0):
        """
        src, dst: LongTensor [B]
        t: FloatTensor [B] or scalar; assumes events are processed in nondecreasing time
        """
        # If t is per-edge, we update sequentially to be exact when a node repeats in batch.
        if t.dim() == 0:
            # scalar time for whole batch
            nodes = torch.unique(torch.cat([src, dst], dim=0))
            self._decay_to(nodes, t)
            self.k[src] += inc
            self.k[dst] += inc
            return

        # per-edge times: do sequential update
        B = src.numel()
        for i in range(B):
            ti = t[i]
            u = src[i].view(1)
            v = dst[i].view(1)
            self._decay_to(u, ti)
            self._decay_to(v, ti)
            self.k[u] += inc
            self.k[v] += inc

    def query(self, nodes: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """
        nodes: LongTensor [...]
        t: FloatTensor scalar or same shape broadcastable to nodes
        Returns k_u(t) for each node, without changing stored k/last_t.
        """
        last = self.last_t[nodes]
        dt = torch.clamp(t.to(last.dtype) - last, min=0.0)
        if self.beta > 0:
            return self.k[nodes] * torch.exp(-self.beta * dt)
        return self.k[nodes]

In [ ]:
import tgn.model.CommunityModel, importlib
importlib.reload(tgn.model.CommunityModel)

from tgn.model.CommunityModel import *
t_min = float(np.min(full_data.timestamps))
num_communities = 5
device = torch.device("mps")

model = CommunityModel(
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    embedding_dim=node_feat.shape[1],
    num_communities=num_communities,
    dropout=0.1,
    init_from_node_features=True,
    t0=t_min,
).to(device)

model.reset_state(t0=t_min)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

In [ ]:
import importlib, sys, time, my_loss
import tgn.utils.utils as utils
importlib.reload(my_loss)
importlib.reload(utils)
from tgn.utils.utils import *
from my_loss import * 
import numpy as np
import torch
import numpy as np
import torch

NUM_EPOCHS = 3
NUM_NEG = 30
lam = 1.0
beta_k = 1
t_min = float(np.min(full_data.timestamp_norm))

# ----- init neg sampler -----
all_nodes = np.unique(np.concatenate([full_data.sources, full_data.destinations])).astype(np.int64)
neg_sampler = NegativeNodeSampler(all_nodes, seed=0, use_seen_pool=True)

for epoch in range(NUM_EPOCHS):
    model.train()
    model.reset_state(t_min)

    activity = ExpKernelActivity(num_nodes=num_nodes, beta=beta_k, device=device, t0=t_min)
    neg_sampler.seen = set()
    for k in range(num_batches):
        start = BATCH_SIZE * k
        end = min(num_instance, BATCH_SIZE * (k + 1))
        sources_batch = full_data.sources[start:end]
        destinations_batch = full_data.destinations[start:end]
        timestamps_batch = full_data.timestamp_norm[start:end]
        edge_idxs_batch = full_data.edge_idxs[start:end]

        B = len(sources_batch)
        if B == 0:
            continue

        src_t = torch.as_tensor(sources_batch, dtype=torch.long, device=device)
        dst_t = torch.as_tensor(destinations_batch, dtype=torch.long, device=device)
        ts_t  = torch.as_tensor(timestamps_batch, dtype=torch.float32, device=device)

        activity.update_endpoints(src_t, dst_t, ts_t)
 
        # ---- sample negatives (node-only) ----
        neg_nodes_np = neg_sampler.sample(B, NUM_NEG)  # [B,R]
        neg_nodes_t  = torch.as_tensor(neg_nodes_np, dtype=torch.long, device=device)


        # ---- compute k BEFORE updating activity with this batch ----
        k_src = activity.query(src_t, ts_t)                    # [B]
        #print(k_src)
        k_neg = activity.query(neg_nodes_t, ts_t.unsqueeze(1)) # [B,R]

        optimizer.zero_grad()

        # ---- forward ----
        p_src, p_dst, p_neg = model.compute_community_prob(
            sources_batch, destinations_batch, neg_nodes_np, timestamps_batch, edge_idxs_batch
        )  # p_neg: [B,R,K]

        # ---- loss ----
        p_prev = torch.full((num_nodes, num_communities), 1.0/num_communities, device=device)
        last_seen_t = torch.full((num_nodes,), t_min, device=device)  # 用 timestamp_norm

        # in each batch, after you build src_t,dst_t,ts_t and got p_src,p_dst,p_neg:
        dt_src = torch.clamp(ts_t - last_seen_t[src_t], min=0.0)
        dt_dst = torch.clamp(ts_t - last_seen_t[dst_t], min=0.0)

        loss, dbg = temporal_modularity_k_weighted_neg_loss(
            p_src=p_src, p_dst=p_dst, p_neg=p_neg,
            k_src=k_src, k_neg=k_neg,
            lam=lam,
            collapse_weight=0.1,
            smooth_weight=0.1,
            smooth_mode="l2",
            p_prev=p_prev,
            nodes_src=src_t,
            nodes_dst=dst_t,
            dt_src=dt_src,
            dt_dst=dt_dst,
            smooth_time_beta=1.0,   # 可选：间隔越大惩罚越小
        )
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            nodes_all = torch.cat([src_t, dst_t], dim=0)
            P_all = torch.cat([p_src, p_dst], dim=0)
            t_all = torch.cat([ts_t, ts_t], dim=0)

            for n, p, tt in zip(nodes_all.tolist(), P_all, t_all):
                p_prev[n] = p.detach()
                last_seen_t[n] = tt


        # ---- update model state from REAL events ----
        if hasattr(model, "update_states_from_events"):
            model.update_states_from_events(
                sources_batch, destinations_batch, timestamps_batch, edge_idxs_batch
            )

        activity.update_endpoints(src_t, dst_t, ts_t)
        neg_sampler.update_seen(sources_batch, destinations_batch)

        # ---- print dbg EVERY batch ----
        # 你也可以换成 logger.info(...)
        print(
            f"[Epoch {epoch+1}][Batch {k+1}/{num_batches}] "
            f"loss={float(loss.item()):.6f} "
            f"pos={dbg['pos_term']:.6f} neg={dbg['neg_term']:.6f} Q={dbg['Q']:.6f} "
            f"avg_k_src={dbg['avg_k_src']:.4f} avg_k_neg={dbg['avg_k_neg']:.4f}"
        )

set()
[Epoch 1][Batch 1/1] loss=-0.327622 pos=0.676656 neg=0.341935 Q=0.334720 avg_k_src=1.1358 avg_k_neg=1.0759
set()
[Epoch 2][Batch 1/1] loss=-0.335292 pos=0.681051 neg=0.338568 Q=0.342483 avg_k_src=1.1358 avg_k_neg=1.0718
set()
[Epoch 3][Batch 1/1] loss=-0.311003 pos=0.660408 neg=0.342206 Q=0.318202 avg_k_src=1.1358 avg_k_neg=1.0731


In [166]:
import json
import numpy as np
import pandas as pd
import torch


def export_probs_to_csv_jodie(
    model,
    full_data,
    BATCH_SIZE: int,
    out_csv_path: str,
    id2node: dict,
    *,
    use_timestamp_norm_for_model: bool = True,
    t0: float | None = None,
):
    """
    Export p_src/p_dst (and argmax labels) for each interaction in chronological order.

    - Model time can use normalized timestamps (timestamp_norm) or raw timestamps.
    - CSV ALWAYS writes raw timestamps to column "timestamp".
    """
    model.eval()

    # --- timestamps for model vs for output ---
    ts_raw_all = full_data.timestamps  # ALWAYS export this
    ts_model_all = full_data.timestamp_norm if use_timestamp_norm_for_model else full_data.timestamps

    # ensure order is chronological w.r.t. model time
    order = np.argsort(ts_model_all, kind="mergesort")

    src_all = full_data.sources[order]
    dst_all = full_data.destinations[order]
    ts_model_all = ts_model_all[order]
    ts_raw_all = ts_raw_all[order]
    eidx_all = full_data.edge_idxs[order]

    # reset state
    if t0 is None:
        t0 = float(np.min(ts_model_all))
    if hasattr(model, "reset_state"):
        model.reset_state(t0=float(t0))

    rows = []

    def _map_id(x):
        x = int(x)
        return id2node[x] if id2node is not None else x

    with torch.no_grad():
        num_instance = len(src_all)
        num_batches = (num_instance + BATCH_SIZE - 1) // BATCH_SIZE

        for k in range(num_batches):
            start = BATCH_SIZE * k
            end = min(num_instance, BATCH_SIZE * (k + 1))

            sources_batch = src_all[start:end]
            destinations_batch = dst_all[start:end]
            ts_model_batch = ts_model_all[start:end]
            ts_raw_batch = ts_raw_all[start:end]
            edge_idxs_batch = eidx_all[start:end]

            B = len(sources_batch)
            if B == 0:
                continue

            # dummy negatives (export doesn't need true negatives)
            neg_nodes = destinations_batch

            p_src, p_dst, _ = model.compute_community_prob(
                sources_batch, destinations_batch, neg_nodes, ts_model_batch, edge_idxs_batch
            )

            p_src_np = p_src.detach().cpu().float().numpy()
            p_dst_np = p_dst.detach().cpu().float().numpy()
            src_commu = p_src_np.argmax(axis=1).astype(np.int64)
            dst_commu = p_dst_np.argmax(axis=1).astype(np.int64)

            for i in range(B):
                rows.append({
                    "source": _map_id(sources_batch[i]),
                    "destination": _map_id(destinations_batch[i]),
                    # ALWAYS write raw timestamp here
                    "timestamp": float(ts_raw_batch[i]),
                    "p_src": json.dumps(p_src_np[i].tolist()),
                    "p_dst": json.dumps(p_dst_np[i].tolist()),
                    "source_commu": int(src_commu[i]),
                    "destination_commu": int(dst_commu[i]),
                })

            model.update_states_from_events(sources_batch, destinations_batch, ts_model_batch, edge_idxs_batch)


    df = pd.DataFrame(rows)
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path} (rows={len(df)})")
    return df

id2node = {idx: node for node, idx in node2id.items()}

export_probs_to_csv_jodie(
    model=model,
    full_data=full_data,
    BATCH_SIZE=BATCH_SIZE,
    out_csv_path="result/FWD_community.csv",
    id2node=id2node,
    use_timestamp_norm_for_model=True,
    t0=float(np.min(full_data.timestamp_norm)),
).head(10)

Saved: result/FWD_community.csv (rows=469)


,source,destination,timestamp,p_src,p_dst,source_commu,destination_commu
0,16,19,5.0,"[1.5475809968194679e-19, 9.572646073262807e-18...","[2.518014253847829e-28, 1.4581248174120006e-23...",2,2
1,6,46,11.0,"[3.124000557592772e-30, 8.100144555100997e-09,...","[7.838982254424771e-22, 3.1267908401355626e-11...",4,4
2,16,28,18.0,"[2.944308548339757e-19, 1.0328055532787058e-17...","[1.0877232884472406e-15, 3.3478390832897276e-0...",2,2
3,5,22,25.0,"[1.87241855866142e-27, 2.3199054339784198e-06,...","[2.8883412587416165e-32, 2.2558940826478135e-1...",4,4
4,8,35,31.0,"[5.185507361772801e-35, 9.705281275024908e-16,...","[3.76524722858762e-12, 1.27657298065742e-08, 8...",4,4
5,12,28,37.0,"[9.792431436176719e-16, 6.331449731497964e-15,...","[4.041982355507687e-15, 3.921428287867457e-06,...",2,2
6,15,41,43.0,"[9.117534780813507e-26, 3.5184449798109085e-10...","[2.654055265223924e-24, 8.299651521603835e-10,...",4,4
7,7,49,52.0,"[3.386664835899627e-21, 1.6029432555603185e-12...","[9.79481769327016e-21, 7.279443839163999e-14, ...",4,4
8,38,43,60.0,"[9.364906227206971e-24, 2.3132080018833934e-12...","[2.9410422188959384e-19, 9.692561336871464e-11...",4,4
9,24,45,63.0,"[0.0, 7.983150851487153e-08, 0.0, 0.0, 0.99999...","[2.9195632023082396e-11, 1.8089731383952312e-0...",4,4


In [184]:
full_data.timestamp_norm

array([  0.        ,   0.21201413,   0.45936396,   0.70671378,
         0.91872792,   1.13074205,   1.34275618,   1.66077739,
         1.9434629 ,   2.04946996,   2.29681979,   2.54416961,
         2.75618375,   2.8975265 ,   3.03886926,   3.18021201,
         3.46289753,   3.63957597,   3.81625442,   4.02826855,
         4.204947  ,   4.48763251,   4.77031802,   4.87632509,
         5.12367491,   5.22968198,   5.4770318 ,   5.75971731,
         6.1130742 ,   6.21908127,   6.36042403,   6.53710247,
         6.67844523,   6.99646643,   7.13780919,   7.45583039,
         7.63250883,   7.77385159,   7.95053004,   8.19787986,
         8.58657244,   8.79858657,   8.93992933,   9.11660777,
         9.57597173,   9.92932862,  10.        ,  10.28268551,
        10.49469965,  10.56537102,  10.84805654,  10.9540636 ,
        11.09540636,  11.37809187,  11.55477032,  11.80212014,
        12.08480565,  12.19081272,  12.29681979,  12.5795053 ,
        12.86219081,  13.07420495,  13.42756184,  13.63

In [229]:
import importlib
import tgn.model.CommunityModel
importlib.reload(tgn.model.CommunityModel)
from tgn.model.CommunityModel import *

import numpy as np
import torch

device = torch.device("mps")
num_communities = 5

ts_reverse = np.ascontiguousarray(full_data.timestamp_norm[::-1])
t_max = float(np.max(full_data.timestamp_norm))

ts_reverse = t_max - ts_reverse

t_min = float(np.min(full_data.timestamp_norm))
bwd_model = CommunityModel(
    node_features=node_feat,
    edge_features=edge_feat,
    device=device,
    embedding_dim=node_feat.shape[1],
    num_communities=num_communities,
    dropout=0.1,
    init_from_node_features=True,
    t0=-t_max,
).to(device)

bwd_model.reset_state(t0=t_min)
opt_bwd = torch.optim.Adam(bwd_model.parameters(), lr=1e-2)
"""

bwd_model.community_projector = model.community_projector

for p in bwd_model.community_projector.parameters():
    p.requires_grad_(False)

opt_bwd = torch.optim.Adam(
    [p for p in bwd_model.parameters() if p.requires_grad],
    lr=1e-2
)

 torch.optim.Adam(
    [p for p in bwd_model.parameters() if p.requires_grad],
    lr=1e-2
)
"""





'\n\nbwd_model.community_projector = model.community_projector\n\nfor p in bwd_model.community_projector.parameters():\n    p.requires_grad_(False)\n\nopt_bwd = torch.optim.Adam(\n    [p for p in bwd_model.parameters() if p.requires_grad],\n    lr=1e-2\n)\n\n torch.optim.Adam(\n    [p for p in bwd_model.parameters() if p.requires_grad],\n    lr=1e-2\n)\n'

In [244]:
import numpy as np
import torch

def train_backward_only(
    model_bwd,
    optimizer,
    full_data,
    *,
    num_epochs: int,
    batch_size: int,
    num_neg: int,
    beta_k: float,
    lam: float,
    device: torch.device,
    num_nodes: int,
    collapse_weight: float = 0.1,
    collapse_from: str = "pos",
    normalize_weights: bool = True
):

    # original time (model uses timestamp_norm)
    ts_all = full_data.timestamp_norm
    if np.any(np.diff(ts_all) < 0):
        raise ValueError("full_data.timestamp_norm must be nondecreasing.")

    t_max = float(np.max(ts_all))
    num_instance = len(full_data.sources)
    num_batches = (num_instance + batch_size - 1) // batch_size

    all_nodes = np.unique(np.concatenate([full_data.sources, full_data.destinations])).astype(np.int64)
    neg_sampler = NegativeNodeSampler(all_nodes, seed=0, use_seen_pool=True)

    for epoch in range(num_epochs):
        model_bwd.train()

        # reversed time starts at 0 (because max - max = 0)
        model_bwd.reset_state(t0=t_min)

        activity_bwd = ExpKernelActivity(num_nodes=num_nodes, beta=beta_k, device=device, t0=t_min)
        neg_sampler.seen = set()

        epoch_loss = 0.0
        epoch_Q = 0.0
        steps = 0

        for k in range(num_batches):
            # take batch from the end
            e = num_instance - batch_size * k
            s = max(0, num_instance - batch_size * (k + 1))

            # reverse inside batch to ensure event-level reverse order
            src = np.ascontiguousarray(full_data.sources[s:e][::-1])
            dst = np.ascontiguousarray(full_data.destinations[s:e][::-1])
            ts  = np.ascontiguousarray(full_data.timestamp_norm[s:e][::-1])
            eidx= np.ascontiguousarray(full_data.edge_idxs[s:e][::-1])

            B = len(src)
            if B == 0:
                continue

            # reversed-time (nonnegative)
            ts_rev = (t_max - ts).astype(np.float64)  # [B]

            src_t = torch.as_tensor(src, dtype=torch.long, device=device)
            dst_t = torch.as_tensor(dst, dtype=torch.long, device=device)
            ts_rev_t = torch.as_tensor(ts_rev, dtype=torch.float32, device=device)

            activity_bwd.update_endpoints(src_t, dst_t, ts_rev_t)

            # negatives (node-only)
            neg_nodes_np = neg_sampler.sample(B, num_neg)  # [B,R]
            neg_nodes_t = torch.as_tensor(neg_nodes_np, dtype=torch.long, device=device)


            # ---- IMPORTANT: query k BEFORE update_endpoints (strict past in t_rev) ----
            k_src = activity_bwd.query(src_t, ts_rev_t)                         # [B]
            k_neg = activity_bwd.query(neg_nodes_t, ts_rev_t.unsqueeze(1))      # [B,R]

            optimizer.zero_grad()

            p_src, p_dst, p_neg = model_bwd.compute_community_prob(
                src, dst, neg_nodes_np, ts_rev, eidx
            )

            loss, dbg = temporal_modularity_k_weighted_neg_loss(
                p_src=p_src, p_dst=p_dst, p_neg=p_neg,
                k_src=k_src, k_neg=k_neg,
                lam=lam,
                symmetric=True,
                normalize_weights=normalize_weights,
                collapse_weight=collapse_weight,
                collapse_from=collapse_from,
            )



            loss.backward()
            print("grad(gru)=", bwd_model.gru.weight_ih.grad.norm().item() if bwd_model.gru.weight_ih.grad is not None else None)
            print("grad(log_decay)=", bwd_model.log_decay.grad.item() if bwd_model.log_decay.grad is not None else None)
            optimizer.step()

            # update model state with REAL events (in reversed-time axis)
            model_bwd.update_states_from_events(src, dst, ts_rev, eidx)

            # update seen pool AFTER step
            activity_bwd.update_endpoints(src_t, dst_t, ts_rev_t)
            neg_sampler.update_seen(src, dst)

            epoch_loss += float(loss.item())
            epoch_Q += float(dbg["Q"])
            steps += 1

        if steps > 0:
            print(
                    f"[BWD][Ep {epoch+1}/{num_epochs}][Batch {k+1}/{num_batches}] "
                    f"loss={float(loss.item()):.6f} Q={dbg['Q']:.6f} "
                    f"pos={dbg['pos_term']:.4f} neg={dbg['neg_term']:.4f} "
                    f"collapse={dbg.get('collapse', 0.0):.4f}"
                )

    return model_bwd

In [234]:
bwd_model = train_backward_only(
    model_bwd=bwd_model,
    optimizer=opt_bwd,
    full_data=full_data,
    num_epochs=500,
    batch_size=BATCH_SIZE,
    num_neg=NUM_NEG,
    beta_k=beta_k,
    lam=lam,
    device=device,
    num_nodes=num_nodes,
    collapse_weight=0.1,
    collapse_from="pos",
    normalize_weights=True
)

[BWD][Ep 1/500][Batch 1/1] loss=-0.279685 Q=0.292804 pos=0.7166 neg=0.4238 collapse=0.1312
[BWD][Ep 2/500][Batch 1/1] loss=-0.277069 Q=0.290313 pos=0.7175 neg=0.4272 collapse=0.1324
[BWD][Ep 3/500][Batch 1/1] loss=-0.281226 Q=0.294441 pos=0.7137 neg=0.4192 collapse=0.1321
[BWD][Ep 4/500][Batch 1/1] loss=-0.271357 Q=0.284528 pos=0.7129 neg=0.4283 collapse=0.1317
[BWD][Ep 5/500][Batch 1/1] loss=-0.279880 Q=0.293101 pos=0.7189 neg=0.4258 collapse=0.1322
[BWD][Ep 6/500][Batch 1/1] loss=-0.266391 Q=0.279593 pos=0.7133 neg=0.4337 collapse=0.1320
[BWD][Ep 7/500][Batch 1/1] loss=-0.272707 Q=0.285876 pos=0.7101 neg=0.4242 collapse=0.1317
[BWD][Ep 8/500][Batch 1/1] loss=-0.273676 Q=0.286737 pos=0.7152 neg=0.4285 collapse=0.1306
[BWD][Ep 9/500][Batch 1/1] loss=-0.266708 Q=0.279895 pos=0.7175 neg=0.4376 collapse=0.1319
[BWD][Ep 10/500][Batch 1/1] loss=-0.279074 Q=0.292247 pos=0.7134 neg=0.4211 collapse=0.1317
[BWD][Ep 11/500][Batch 1/1] loss=-0.283603 Q=0.296790 pos=0.7145 neg=0.4177 collapse=0.13

In [233]:
import json
import numpy as np

import torch
import pandas as pd

def export_probs_to_csv_bwd(
    bwd_model,
    full_data,
    BATCH_SIZE: int,
    out_csv_path: str,
    id2node: dict,
    *,
    use_timestamp_norm_for_model: bool = True,
    t0_rev: float | None = None,   # bwd 的 t0（在 t_rev 轴上）
):
    """
    Export p_src/p_dst for a BACKWARD-trained model.

    Output CSV is in chronological order (in raw timestamp), but the model state is advanced
    by iterating events in REVERSE order using reversed-time:
        t_rev = t_max - t_model

    - Model time uses timestamp_norm (recommended) or raw timestamps.
    - CSV ALWAYS writes raw timestamps to column "timestamp".
    """
    bwd_model.eval()

    # --- timestamps for model vs for output ---
    ts_raw_all = full_data.timestamps
    ts_model_all = full_data.timestamp_norm if use_timestamp_norm_for_model else full_data.timestamps

    # ensure chronological order w.r.t model time
    order = np.argsort(ts_model_all, kind="mergesort")

    src_all = full_data.sources[order]
    dst_all = full_data.destinations[order]
    ts_model_all = ts_model_all[order]
    ts_raw_all = ts_raw_all[order]
    eidx_all = full_data.edge_idxs[order]

    N = len(src_all)
    if N == 0:
        df = pd.DataFrame(columns=[
            "source", "destination", "timestamp", "p_src", "p_dst", "source_commu", "destination_commu"
        ])
        df.to_csv(out_csv_path, index=False)
        print(f"Saved: {out_csv_path} (rows=0)")
        return df

    t_max = float(np.max(ts_model_all))

    # reversed-time axis for bwd model
    ts_rev_all = (t_max - ts_model_all).astype(np.float64)

    # reset bwd state
    if t0_rev is None:
        t0_rev = float(np.min(ts_rev_all))  # should be 0.0
    if hasattr(bwd_model, "reset_state"):
        bwd_model.reset_state(t0=float(t0_rev))

    def _map_id(x):
        x = int(x)
        return id2node[x] if id2node is not None else x

    # store outputs aligned to chronological index i
    # (we will fill these while running backward, then write CSV in chronological order)
    p_src_out = None
    p_dst_out = None

    with torch.no_grad():
        num_batches = (N + BATCH_SIZE - 1) // BATCH_SIZE

        # iterate from end to start
        for k in range(num_batches):
            end = N - BATCH_SIZE * k
            start = max(0, N - BATCH_SIZE * (k + 1))

            # take slice [start:end], then reverse inside batch for correct backward progression
            src_b = np.ascontiguousarray(src_all[start:end][::-1])
            dst_b = np.ascontiguousarray(dst_all[start:end][::-1])
            ts_rev_b = np.ascontiguousarray(ts_rev_all[start:end][::-1])
            eidx_b = np.ascontiguousarray(eidx_all[start:end][::-1])

            B = len(src_b)
            if B == 0:
                continue

            # dummy negatives
            neg_nodes = dst_b

            p_src, p_dst, _ = bwd_model.compute_community_prob(
                src_b, dst_b, neg_nodes, ts_rev_b, eidx_b
            )

            p_s = p_src.detach().cpu().float().numpy()
            p_d = p_dst.detach().cpu().float().numpy()

            # init storage (infer K from first batch)
            if p_src_out is None:
                K = p_s.shape[1]
                p_src_out = np.zeros((N, K), dtype=np.float32)
                p_dst_out = np.zeros((N, K), dtype=np.float32)

            # ps/pd are in reversed order within this batch; flip back to match chronological indices
            p_src_out[start:end] = p_s[::-1]
            p_dst_out[start:end] = p_d[::-1]

            # advance bwd state using reversed-time and reversed event order
            if hasattr(bwd_model, "update_states_from_events"):
                bwd_model.update_states_from_events(src_b, dst_b, ts_rev_b, eidx_b)

    # build rows in chronological order
    rows = []
    for i in range(N):
        p_s = p_src_out[i]
        p_d = p_dst_out[i]
        rows.append({
            "source": _map_id(src_all[i]),
            "destination": _map_id(dst_all[i]),
            "timestamp": float(ts_raw_all[i]),
            "p_src": json.dumps(p_s.tolist()),
            "p_dst": json.dumps(p_d.tolist()),
            "source_commu": int(np.argmax(p_s)),
            "destination_commu": int(np.argmax(p_d)),
        })

    df = pd.DataFrame(rows)
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path} (rows={len(df)})")
    return df


id2node = {idx: node for node, idx in node2id.items()}

df_bwd = export_probs_to_csv_bwd(
    bwd_model=bwd_model,
    full_data=full_data,
    BATCH_SIZE=BATCH_SIZE,
    out_csv_path="result/BWD_community.csv",
    id2node=id2node,
    use_timestamp_norm_for_model=True,
    t0_rev=0.0,
)
df_bwd.head(10)

Saved: result/BWD_community.csv (rows=469)


,source,destination,timestamp,p_src,p_dst,source_commu,destination_commu
0,16,19,5.0,"[0.9995297193527222, 0.00046939606545493007, 3...","[0.0955558717250824, 0.904147207736969, 2.6824...",0,1
1,6,46,11.0,"[0.9997599720954895, 0.00023962781415320933, 1...","[0.9998512268066406, 0.0001484762760810554, 2....",0,0
2,16,28,18.0,"[0.9995124340057373, 0.0004866453818976879, 4....","[0.9932002425193787, 0.006413585506379604, 9.0...",0,0
3,5,22,25.0,"[0.009340778924524784, 0.990646481513977, 9.97...","[0.03900735825300217, 0.9606421589851379, 6.85...",1,1
4,8,35,31.0,"[0.9999336004257202, 6.62976162857376e-05, 1.5...","[0.8611882925033569, 0.13853932917118073, 1.16...",0,0
5,12,28,37.0,"[0.0076917619444429874, 0.9910280108451843, 9....","[0.9926943778991699, 0.006876042112708092, 1.4...",1,0
6,15,41,43.0,"[0.9998018145561218, 0.00019808104843832552, 3...","[0.9997923970222473, 0.00020558334654197097, 1...",0,0
7,7,49,52.0,"[0.136091947555542, 0.8592244386672974, 8.4962...","[0.9996053576469421, 0.0003934655978810042, 5....",1,0
8,38,43,60.0,"[0.917746901512146, 0.08113177120685577, 2.363...","[0.9977645874023438, 0.00215689348988235, 5.84...",0,0
9,24,45,63.0,"[0.9999122619628906, 8.756850729696453e-05, 3....","[0.9563586115837097, 0.043572574853897095, 2.5...",0,0


In [235]:
bwd_model.load_state_dict(model.state_dict(), strict=False)

<All keys matched successfully>

In [236]:
# 共享 projector
bwd_model.community_projector = model.community_projector

# 先冻结 bwd 全部参数
for p in bwd_model.parameters():
    p.requires_grad_(False)

# 冻结 projector（保持社区语义不变）
for p in bwd_model.community_projector.parameters():
    p.requires_grad_(False)

# ---- 解冻 bwd 需要学习的“动力学部分” ----
# 1) GRU（事件更新）
for p in bwd_model.gru.parameters():
    p.requires_grad_(True)

# 2) log_decay（时间衰减标量）
bwd_model.log_decay.requires_grad_(True)

# 3) time_encoder（可选：如果 TimeEncode 有可学习参数）
for p in bwd_model.time_encoder.parameters():
    p.requires_grad_(True)

In [240]:
opt_bwd = torch.optim.Adam([p for p in bwd_model.parameters() if p.requires_grad], lr=1e-2)

In [242]:
trainable = [(n,p) for n,p in bwd_model.named_parameters() if p.requires_grad]
print("trainable:", [n for n,_ in trainable])

trainable: ['log_decay', 'time_encoder.w.weight', 'time_encoder.w.bias', 'gru.weight_ih', 'gru.weight_hh', 'gru.bias_ih', 'gru.bias_hh']


In [245]:
bwd_model = train_backward_only(
    model_bwd=bwd_model,
    optimizer=opt_bwd,
    full_data=full_data,
    num_epochs=10,
    batch_size=BATCH_SIZE,
    num_neg=NUM_NEG,
    beta_k=beta_k,
    lam=lam,
    device=device,
    num_nodes=num_nodes,
    collapse_weight=0.1,
    collapse_from="pos",
    normalize_weights=True
)

grad(gru)= None
grad(log_decay)= 0.030524183064699173
[BWD][Ep 1/10][Batch 1/1] loss=-0.080059 Q=0.089182 pos=0.4180 neg=0.3289 collapse=0.0912
grad(gru)= None
grad(log_decay)= -0.04077008366584778
[BWD][Ep 2/10][Batch 1/1] loss=-0.091982 Q=0.101139 pos=0.4244 neg=0.3232 collapse=0.0916
grad(gru)= None
grad(log_decay)= 0.03154545649886131
[BWD][Ep 3/10][Batch 1/1] loss=-0.099772 Q=0.108625 pos=0.4294 neg=0.3207 collapse=0.0885
grad(gru)= None
grad(log_decay)= -0.05330602824687958
[BWD][Ep 4/10][Batch 1/1] loss=-0.087066 Q=0.095995 pos=0.4225 neg=0.3265 collapse=0.0893
grad(gru)= None
grad(log_decay)= 0.012666404247283936
[BWD][Ep 5/10][Batch 1/1] loss=-0.076348 Q=0.085291 pos=0.4156 neg=0.3303 collapse=0.0894
grad(gru)= None
grad(log_decay)= 0.06348584592342377
[BWD][Ep 6/10][Batch 1/1] loss=-0.092766 Q=0.101502 pos=0.4237 neg=0.3222 collapse=0.0874
grad(gru)= None
grad(log_decay)= -0.053874529898166656
[BWD][Ep 7/10][Batch 1/1] loss=-0.085336 Q=0.093921 pos=0.4148 neg=0.3209 collapse=

In [ ]:
import numpy as np
import torch

@torch.no_grad()
def infer_forward_endpoints(
    model_fwd,
    full_data,
    *,
    batch_size: int,
):
    model_fwd.eval()
    ts = full_data.timestamp_norm
    if np.any(np.diff(ts) < 0):
        raise ValueError("full_data.timestamp_norm must be nondecreasing.")

    t_min = float(np.min(ts))
    model_fwd.reset_state(t0=t_min)

    N = len(full_data.sources)
    # infer K by one forward
    K = model_fwd.num_communities

    p_src_all = np.zeros((N, K), dtype=np.float32)
    p_dst_all = np.zeros((N, K), dtype=np.float32)

    for s in range(0, N, batch_size):
        e = min(N, s + batch_size)
        src = full_data.sources[s:e]
        dst = full_data.destinations[s:e]
        tsb = full_data.timestamp_norm[s:e]
        eidx= full_data.edge_idxs[s:e]

        neg = dst  # dummy
        p_src, p_dst, _ = model_fwd.compute_community_prob(src, dst, neg, tsb, eidx)

        p_src_all[s:e] = p_src.detach().cpu().numpy()
        p_dst_all[s:e] = p_dst.detach().cpu().numpy()

        model_fwd.update_states_from_events(src, dst, tsb, eidx)

    return p_src_all, p_dst_all


@torch.no_grad()
def infer_backward_endpoints(
    model_bwd,
    full_data,
    *,
    batch_size: int,
):
    model_bwd.eval()
    ts = full_data.timestamp_norm
    if np.any(np.diff(ts) < 0):
        raise ValueError("full_data.timestamp_norm must be nondecreasing.")

    t_max = float(np.max(ts))
    N = len(full_data.sources)
    K = model_bwd.num_communities

    p_src_all = np.zeros((N, K), dtype=np.float32)
    p_dst_all = np.zeros((N, K), dtype=np.float32)

    # reset in reversed-time axis
    model_bwd.reset_state(t0=-t_max)

    # traverse from end to start
    for e in range(N, 0, -batch_size):
        s = max(0, e - batch_size)

        # reverse inside batch for correct backward progression
        src = np.ascontiguousarray(full_data.sources[s:e][::-1])
        dst = np.ascontiguousarray(full_data.destinations[s:e][::-1])
        tsb = np.ascontiguousarray(full_data.timestamp_norm[s:e][::-1])
        eidx= np.ascontiguousarray(full_data.edge_idxs[s:e][::-1])

        ts_rev = -tsb
        neg = dst  # dummy

        p_src, p_dst, _ = model_bwd.compute_community_prob(src, dst, neg, ts_rev, eidx)

        # outputs are in reversed order; flip back to write into [s:e]
        ps = p_src.detach().cpu().numpy()[::-1]
        p_d = p_dst.detach().cpu().numpy()[::-1]
        p_src_all[s:e] = ps
        p_dst_all[s:e] = pd

        model_bwd.update_states_from_events(src, dst, ts_rev, eidx)

    return p_src_all, p_dst_all

import numpy as np

def fuse_probs(p_fwd: np.ndarray, p_bwd: np.ndarray, mode: str = "poe", eps: float = 1e-12) -> np.ndarray:
    """
    p_fwd, p_bwd: [N,K] probabilities
    mode:
      - "avg": arithmetic mean
      - "poe": product-of-experts => softmax(log p_fwd + log p_bwd)
    """
    if mode == "avg":
        p = 0.5 * (p_fwd + p_bwd)
        return p / (p.sum(axis=1, keepdims=True) + eps)

    if mode == "poe":
        lf = np.log(np.clip(p_fwd, eps, 1.0))
        lb = np.log(np.clip(p_bwd, eps, 1.0))
        logits = lf + lb
        logits = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(logits)
        return exp / (exp.sum(axis=1, keepdims=True) + eps)

    raise ValueError("mode must be 'avg' or 'poe'")

import json
import pandas as pd

def export_fused_edge_csv(
    full_data,
    id2node: dict,
    out_csv_path: str,
    p_src_fused: np.ndarray,
    p_dst_fused: np.ndarray,
):
    N = len(full_data.sources)
    rows = []
    for i in range(N):
        rows.append({
            "source": id2node[int(full_data.sources[i])],
            "destination": id2node[int(full_data.destinations[i])],
            "timestamp": float(full_data.timestamps[i]),  # raw
            "p_src": json.dumps(p_src_fused[i].tolist()),
            "p_dst": json.dumps(p_dst_fused[i].tolist()),
            "source_commu": int(p_src_fused[i].argmax()),
            "destination_commu": int(p_dst_fused[i].argmax()),
        })
    df = pd.DataFrame(rows)
    df.to_csv(out_csv_path, index=False)
    print(f"Saved: {out_csv_path} (rows={len(df)})")
    return df



